In [1]:
import torch
import torchvision
import numpy
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.datasets
# import matplolib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [2]:
print(device)

cpu


In [3]:
#loader dataset
transform = transforms.Compose([
    transforms.Resize(28),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.1307],
                         std=[0.0307])
])

dataset = './data/'

train_set = torchvision.datasets.MNIST(root=dataset, train=True, download=True, transform=transform)
test_set  = torchvision.datasets.MNIST(root=dataset, train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size = 32, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size = 32, shuffle=False)

In [4]:
# mlp and cnn class
class SimpleMLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)
        self.flatten = nn.Flatten()
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)

        return x

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        # gray scale image = 1 in_channel, out_channels = 32 features
        # kernel_size = 3 and padding = (kernel_size - 1)/2
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.maxpool1 = nn.MaxPool2d(kernel_size=2, stride=2) # 1 x 28 x 28 -> 32 x 14 x 14

        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.maxpool2 = nn.MaxPool2d(kernel_size=2, stride=2) # 64 x 7 x 7

        self.flatten = nn.Flatten()
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.maxpool1(self.relu(self.conv1(x)))
        x = self.maxpool2(self.relu(self.conv2(x)))

        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [5]:
def train_model(model, train_data, num_epochs, lr=0.001):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()

    for epoch in range(num_epochs):
        num_batch = 0
        epoch_loss = 0
        correct, total = 0, 0

        for batch_x, batch_y in train_data:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            logits = model.forward(batch_x)
            loss = loss_fn(logits, batch_y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            num_batch += 1
            epoch_loss += loss.item()
            pred = logits.argmax(dim=1)
            correct += (batch_y == pred).sum().item()
            total += batch_y.size(0)

        avg_loss = epoch_loss / num_batch
        accuracy = correct / total
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}')

In [6]:
# # train Simple MLP
# print('Training MLP...')
# mlp_model = SimpleMLP(input_size = 28*28*1, hidden_size=32, output_size=10)
# mlp_model.to(device)
# train_model(mlp_model, train_data=train_loader, num_epochs=10, lr=0.001)

In [7]:
# train CNN model
print('Training CNN...')
cnn_model = CNN()
cnn_model.to(device=device)
train_model(cnn_model, train_data=train_loader, num_epochs=5, lr=0.001)

Training CNN...
Epoch [1/5], Loss: 0.1174, Accuracy: 0.9643
Epoch [2/5], Loss: 0.0459, Accuracy: 0.9862
Epoch [3/5], Loss: 0.0343, Accuracy: 0.9893
Epoch [4/5], Loss: 0.0259, Accuracy: 0.9912
Epoch [5/5], Loss: 0.0210, Accuracy: 0.9936


In [8]:
# eval
def predict(model, test_data = test_loader, device = device):
    model.eval()
    all_prediction = []
    all_labels = []

    with torch.no_grad():
        for batch_x, batch_y in test_data:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            logits = model(batch_x)
            preds = logits.argmax(dim=1)
            all_prediction.append(preds.cpu().numpy())
            all_labels.append(preds.cpu().numpy())

    y_preds = numpy.concatenate(all_prediction)
    y_true  = numpy.concatenate(all_labels)
    return y_preds, y_true

In [9]:
# mlp_pred, mlp_true = predict(mlp_model)

# accuracy = [int(i == j) for i,j in zip(mlp_pred,mlp_true)]
# print(mlp_pred, mlp_true)
# print(f'Accuracy: {(sum(accuracy)/len(mlp_pred))*100}%')

In [10]:
cnn_pred, cnn_true = predict(cnn_model)

accuracy = [int(i == j) for i,j in zip(cnn_pred,cnn_true)]
print(cnn_pred, cnn_true)
print(f'Accuracy: {(sum(accuracy)/len(cnn_pred))*100}%')

[7 2 1 ... 4 5 6] [7 2 1 ... 4 5 6]
Accuracy: 100.0%


### Load Model's Weights

In [11]:
torch.save(cnn_model.state_dict(), "cnn.pth")

In [12]:
import matplotlib.pyplot as plt
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def predict_image(model, image, device=device):
    model.eval()

    image = image.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(image)
        prediction = logits.argmax(dim=1)

    return prediction.item()


In [1]:
from PIL import Image
import torchvision.transforms as transforms

model = CNN()

model.load_state_dict(torch.load('cnn.pth'))
model.eval()

image = Image.open("drawing.png").convert("L")

transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor()
])

image = transform(image)

image, label = test_set[9]

print("Actual:", label)
print("Predicted:", predict_image(model, image))

plt.imshow(image.squeeze(), cmap='gray')
plt.axis('off')
plt.show()

NameError: name 'CNN' is not defined